# Train one YOLO model on Kaggle

Trains a single model variant and writes the run folder to `/kaggle/working/runs/`.
When the notebook is committed via **Save Version → Save & Run All**, Kaggle snapshots
`/kaggle/working/` as the notebook's output — that's how runs persist across sessions.

**One-time setup before running:**
1. Upload your dataset zip as a Kaggle Dataset (kaggle.com/datasets → New Dataset).
   - Upload both `dataset.zip` and `dataset.meta.json` from `data_prep/build/`.
   - Note the slug (e.g. `your-username/safety-equipment-yolo-v1`).
2. In the Kaggle notebook: right sidebar → **+ Add Data** → search for and attach your dataset.
3. Right sidebar → **Settings** → toggle **Internet** ON, **Accelerator** = GPU T4 x2 or P100.
4. Set `INPUT_DATASET_DIR` below to the path where the zip ended up under `/kaggle/input/`.

**To resume a previous run** (e.g. after a 9h session timeout):
- After the previous session, click **Save Version → Save & Run All** so its output is captured.
- In the new session, **+ Add Data** → **Your Work** → add that previous notebook version's output.
- Copy its runs into /kaggle/working/runs/ (cell below does this), then set `RESUME=True`.

In [ ]:
# === EDIT THESE PER SESSION ===
REPO_URL    = 'https://github.com/tahmid013/yolo.git'
REPO_BRANCH = 'main'

# Path inside /kaggle/input/ where dataset.zip lives. After 'Add Data' the
# attached dataset appears at /kaggle/input/<dataset-slug>/. Edit accordingly.
INPUT_DATASET_DIR = '/kaggle/input/safety-equipment-yolo-v1'

# v11 already trained by the reference team (imported via analysis/import_reference.py).
# This pipeline is now focused on training the v12 family. Cycle through these 4:
#   yolo12n (start here) -> yolo12s -> yolo12m -> yolo12l
MODEL  = 'yolo12n'
CONFIG = 'configs/yolo12n.yaml'
RESUME = False                         # True to continue the latest run in PREV_RUNS_DIR
PREV_RUNS_DIR = None                   # e.g. '/kaggle/input/<prev-notebook-output>/runs'
CHECKPOINT_EVERY = 10                  # sync weights/results every N epochs (0 to disable)
# ==============================

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil, subprocess, sys

# Step out of /kaggle/working/code before deleting it — otherwise git
# inherits a phantom CWD from a previous session and fails.
os.chdir('/kaggle/working')

if os.path.isdir('/kaggle/working/code'):
    shutil.rmtree('/kaggle/working/code')
res = subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, '/kaggle/working/code'],
    capture_output=True, text=True,
)
if res.returncode != 0:
    print('git stdout:', res.stdout)
    print('git stderr:', res.stderr)
    raise RuntimeError(f'git clone failed (exit {res.returncode})')
os.chdir('/kaggle/working/code')
if '/kaggle/working/code' not in sys.path:
    sys.path.insert(0, '/kaggle/working/code')
# purge any stale `pipeline` cache from previous kernel state
for mod in [m for m in list(sys.modules) if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[mod]

print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
!pip install -q -r requirements.txt
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
from pathlib import Path
from pipeline.dataset import ensure_dataset

zip_path  = Path(INPUT_DATASET_DIR) / 'dataset.zip'
meta_path = Path(INPUT_DATASET_DIR) / 'dataset.meta.json'
if not zip_path.exists():
    raise FileNotFoundError(
        f'dataset.zip not found at {zip_path}.\n'
        f'Available under /kaggle/input/:\n  ' + '\n  '.join(sorted(os.listdir("/kaggle/input")))
    )

data_yaml = ensure_dataset(
    zip_path=zip_path,
    meta_path=meta_path,
    target='/kaggle/working/dataset',
)
print('data.yaml:', data_yaml)
DATASET_META = meta_path

In [ ]:
# If RESUME=True and PREV_RUNS_DIR is set, copy previous runs into /kaggle/working/runs/
# so the trainer can find the last checkpoint for MODEL.
from pathlib import Path
import shutil

LOCAL_RUNS = Path('/kaggle/working/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

if RESUME:
    if not PREV_RUNS_DIR:
        raise RuntimeError('RESUME=True but PREV_RUNS_DIR is not set')
    prev = Path(PREV_RUNS_DIR)
    if not prev.is_dir():
        raise FileNotFoundError(f'PREV_RUNS_DIR not found: {prev}')
    for sub in prev.iterdir():
        if sub.is_dir() and sub.name.startswith(f'{MODEL}_'):
            dst = LOCAL_RUNS / sub.name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(sub, dst)
            print('staged for resume:', dst)

In [ ]:
from pipeline.train import run as train_run

# On Kaggle we treat /kaggle/working/runs/ as BOTH the local and 'remote' runs dir,
# since /kaggle/working/ is what gets persisted when you Save Version. The atomic
# copy step still runs but is a no-op rename within the same filesystem.
run_dir = train_run(
    model=MODEL,
    config=CONFIG,
    drive_runs_dir='/kaggle/working/runs',
    local_runs_dir='/kaggle/working/runs_tmp',
    data_yaml=data_yaml,
    dataset_meta_path=DATASET_META,
    base_config='configs/base.yaml',
    resume=RESUME,
    checkpoint_every=CHECKPOINT_EVERY,
)
print('Run saved to:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_payload = eval_run(
    run_dir=run_dir,
    data_yaml=data_yaml,
    drive_runs_dir='/kaggle/working/runs',
)
print('test mAP50:    ', eval_payload['overall']['mAP50'])
print('test mAP50-95: ', eval_payload['overall']['mAP50_95'])

In [ ]:
# Tidy: drop the unzipped dataset and the /content/code dir from /kaggle/working/
# so the Save Version output only contains runs/ (smaller download).
import shutil
for p in ['/kaggle/working/dataset', '/kaggle/working/code',
          '/kaggle/working/runs_tmp']:
    if os.path.isdir(p):
        shutil.rmtree(p)
print('working contents:')
!ls -la /kaggle/working/

## Next step: Save Version

Click **Save Version → Save & Run All (Commit)** in the top right. Kaggle will:
- Re-run the notebook end-to-end on a fresh runtime (you can close your browser).
- Capture `/kaggle/working/` as the version's downloadable output.

After it finishes, open the notebook version → **Output** tab → download `runs/` for
local analysis, or attach this version's output to a future notebook (for resume).